# RetinaAI Triage — training on Kaggle

**Before running:**
1. *Settings → Accelerator → GPU (T4 or P100)*
2. *Settings → Internet → On* (needed for `git clone` and pretrained ImageNet weights)
3. *Add Input → Competitions →* **APTOS 2019 Blindness Detection** (accept the competition rules once on its page)

Research/educational use only.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO = "https://github.com/YOUR_GITHUB_USER/retinaai-triage.git"   # <-- change
!git clone -q $REPO /kaggle/working/retinaai-triage
%cd /kaggle/working/retinaai-triage
!ls /kaggle/input/aptos2019-blindness-detection | head

## Stage 0 — smoke test (~3 min): does the whole pipeline run?

In [ ]:
!PYTHONPATH=src python -m retinaai.train --max_samples 600 --epochs 2 --img_size 256 \
    --batch_size 32 --output_dir /kaggle/working/outputs/smoke

## Stage 1 — baseline on full APTOS (~15–25 min on T4)
First epoch also builds the image cache (a few minutes, once).

In [ ]:
!PYTHONPATH=src python -m retinaai.train --epochs 10 --img_size 384 --batch_size 16 \
    --smartphone_aug_p 0.3 --output_dir /kaggle/working/outputs/baseline

In [ ]:
import json, pandas as pd
h = pd.DataFrame(json.load(open("/kaggle/working/outputs/baseline/history.json")))
display(h[["epoch","train_loss","qwk","accuracy","referable_sensitivity","referable_specificity","referable_auc"]].round(3))
best = h.loc[h.qwk.idxmax()]
print("Confusion matrix (rows=true grade, cols=predicted) at best epoch:")
display(pd.DataFrame(best.confusion_matrix))

## Stage 2 — robustness to smartphone-like images (staged training)
Fine-tune the baseline with stronger degradation. Compare metrics with Stage 1.

In [ ]:
!PYTHONPATH=src python -m retinaai.train --init_from /kaggle/working/outputs/baseline/model_inference.pt \
    --epochs 4 --lr 1e-4 --warmup_epochs 0.2 --smartphone_aug_p 0.6 \
    --output_dir /kaggle/working/outputs/robust

## Calibrate the image-quality thresholds

In [ ]:
!PYTHONPATH=src python scripts/quality_report.py \
    --images /kaggle/input/aptos2019-blindness-detection/train_images \
    --limit 300 --degrade --out /kaggle/working/outputs/quality_report.csv

## Try the predictor (grade + uncertainty + quality + triage + heatmap)

In [ ]:
import sys, base64, io, glob
sys.path.insert(0, "src")
from PIL import Image
from retinaai.predict import Predictor
p = Predictor("/kaggle/working/outputs/baseline/model_inference.pt")
img_path = sorted(glob.glob("/kaggle/input/aptos2019-blindness-detection/train_images/*.png"))[0]
res = p.predict(img_path)
display(Image.open(io.BytesIO(base64.b64decode(res["heatmap_png_base64"]))).resize((320, 320)))
res["heatmap_png_base64"] = "<omitted>"
res

## Share the weights with the team
`model_inference.pt` (~20 MB) is in the notebook's **Output** tab.
Download it and attach it to a **GitHub Release** of the repo (do not commit `.pt` files).
Backend then runs: `RETINAAI_CHECKPOINT=weights/model_inference.pt uvicorn api.app:app`